# LightGBM: подбор гиперпараметров через Optuna

- Подбор на train 12–31 / valid 32 при фиксированном `learning_rate = 0.1`. Число деревьев не подбирается, его даёт ранняя остановка.
- Плохие trials обрываются досрочно: каждые 10 итераций RMSE на valid сравнивается с медианой прошлых trials на той же итерации.
- Study хранится в SQLite (`data/optuna/lgbm.db`): если прервать ноутбук, следующий запуск продолжит с того же места.
  Если поменялись признаки или разбиение, поменяй `STUDY_NAME`, иначе старые trials смешаются с новыми.
- В конце лучшие параметры обучаются с `learning_rate = 0.05`.

In [3]:
import json
from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd

import scripts.feature_engineering as feature
from scripts.evaluate import CLIP, calculate_metrics, rmse

In [4]:
df = pd.read_csv('data/gold/train.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')
df_items = pd.read_csv('data/landing/items.csv')
df_shops = pd.read_csv('data/landing/shops.csv')
df_sales = pd.read_csv('data/landing/sales_train.csv')

df = feature.add_categories(df, df_cat, df_items)
df = feature.add_cities(df, df_shops)
df = feature.add_target_lags(df, (1, 2, 3, 4, 5, 6, 12))
df = feature.add_lag_stats(df)
df = feature.add_mean_lags(df)
df = feature.add_age_features(df)
df = feature.add_price_features(df, df_sales, df_items)
df = feature.add_calendar(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10884554 entries, 0 to 10884553
Data columns (total 34 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date_block_num        int64  
 1   shop_id               int64  
 2   item_id               int64  
 3   target                float64
 4   item_category_id      uint8  
 5   category_type_id      uint8  
 6   item_type_id          uint8  
 7   city_id               uint8  
 8   target_lag_1          float32
 9   target_lag_2          float32
 10  target_lag_3          float32
 11  target_lag_4          float32
 12  target_lag_5          float32
 13  target_lag_6          float32
 14  target_lag_12         float32
 15  roll3                 float32
 16  roll6                 float32
 17  diff_1_2              float32
 18  item_mean_lag_1       float32
 19  shop_mean_lag_1       float32
 20  cat_mean_lag_1        float32
 21  shop_cat_mean_lag_1   float32
 22  city_item_mean_lag_1  float32
 23  pair_age        

In [5]:
CAT_FEATURES = ['shop_id', 'item_category_id', 'category_type_id', 'item_type_id', 'city_id']
FEATURES = [c for c in df.columns if c not in ('target', 'ID', 'date_block_num', 'item_id')]
print(FEATURES)

['shop_id', 'item_category_id', 'category_type_id', 'item_type_id', 'city_id', 'target_lag_1', 'target_lag_2', 'target_lag_3', 'target_lag_4', 'target_lag_5', 'target_lag_6', 'target_lag_12', 'roll3', 'roll6', 'diff_1_2', 'item_mean_lag_1', 'shop_mean_lag_1', 'cat_mean_lag_1', 'shop_cat_mean_lag_1', 'city_item_mean_lag_1', 'pair_age', 'pair_since_last', 'item_age', 'item_is_new', 'item_price_last', 'item_price_ratio', 'price_rel_cat', 'month', 'month_sin', 'month_cos', 'days']


In [6]:
m = df['date_block_num']
train, valid = df[(m >= 12) & (m <= 31)], df[m == 32]
tr_s = train.sample(500_000, random_state=42)  # только для кривой train в финальной модели

## Пространство поиска и целевая функция

In [7]:
STUDY_NAME = 'lgbm_v1'
OPTUNA_DIR = Path('data/optuna')
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

FIXED: dict[str, Any] = {
    'n_estimators': 3000, 'learning_rate': 0.1, 'metric': 'rmse',
    'subsample_freq': 1, 'random_state': 42, 'n_jobs': -1, 'verbose': -1,
}


def suggest_params(trial: optuna.Trial) -> dict[str, Any]:
    params: dict[str, Any] = {
        'objective': trial.suggest_categorical('objective', ['regression', 'tweedie']),
        'num_leaves': trial.suggest_int('num_leaves', 31, 511, log=True),          # сложность дерева
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 3000, log=True),  # главная регуляризация
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),     # доля признаков на дерево
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),                   # доля строк на дерево
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),     # L2 на значения листьев
        'cat_smooth': trial.suggest_float('cat_smooth', 1.0, 100.0, log=True),     # сглаживание категорий
    }
    if params['objective'] == 'tweedie':
        params['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return params


def pruning_callback(trial: optuna.Trial, period: int = 10):
    """Каждые period итераций сообщает Optuna RMSE на valid и обрывает trial, если он хуже медианы."""
    def _callback(env: lgb.callback.CallbackEnv) -> None:
        if env.iteration % period:
            return
        score = next(r[2] for r in env.evaluation_result_list if r[0] == 'valid')
        trial.report(score, env.iteration)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return _callback


def objective(trial: optuna.Trial) -> float:
    model = lgb.LGBMRegressor(**FIXED, **suggest_params(trial))
    model.fit(
        train[FEATURES], train['target'],
        eval_X=(valid[FEATURES],), eval_y=(valid['target'],), eval_names=['valid'],
        categorical_feature=CAT_FEATURES,
        callbacks=[lgb.early_stopping(100, verbose=False), pruning_callback(trial)],
    )
    trial.set_user_attr('best_iteration', model.best_iteration_)
    # та же метрика, что в calculate_metrics: прогноз обрезан до [0, 20]
    return rmse(valid['target'], np.clip(model.predict(valid[FEATURES]), *CLIP))

In [8]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{OPTUNA_DIR / "lgbm.db"}',
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30),
)
if len(study.trials) == 0:
    study.enqueue_trial({
        'objective': 'regression', 'num_leaves': 63, 'min_child_samples': 200,
        'colsample_bytree': 0.8, 'subsample': 0.8, 'reg_lambda': 1.0, 'cat_smooth': 10.0,
    })

study.optimize(objective, n_trials=40, timeout=None, show_progress_bar=True)

[I 2026-10-30 21:04:41,845] A new study created in RDB with name: lgbm_v1
Best trial: 0. Best value: 0.852583:   2%|▎         | 1/40 [00:28<18:38, 28.69s/it]

[I 2026-10-30 21:05:10,549] Trial 0 finished with value: 0.8525826760216981 and parameters: {'objective': 'regression', 'num_leaves': 63, 'min_child_samples': 200, 'colsample_bytree': 0.8, 'subsample': 0.8, 'reg_lambda': 1.0, 'cat_smooth': 10.0}. Best is trial 0 with value: 0.8525826760216981.


Best trial: 0. Best value: 0.852583:   5%|▌         | 2/40 [01:17<25:42, 40.59s/it]

[I 2026-10-30 21:05:59,476] Trial 1 finished with value: 0.8530656964042436 and parameters: {'objective': 'tweedie', 'num_leaves': 240, 'min_child_samples': 398, 'colsample_bytree': 0.4936111842654619, 'subsample': 0.5779972601681014, 'reg_lambda': 0.001819899369726822, 'cat_smooth': 53.99484409787433, 'tweedie_variance_power': 1.580892009394567}. Best is trial 0 with value: 0.8525826760216981.


Best trial: 0. Best value: 0.852583:   8%|▊         | 3/40 [02:33<35:06, 56.92s/it]

[I 2026-10-30 21:07:15,815] Trial 2 finished with value: 0.8536122082884661 and parameters: {'objective': 'regression', 'num_leaves': 470, 'min_child_samples': 1290, 'colsample_bytree': 0.5274034664069657, 'subsample': 0.5909124836035503, 'reg_lambda': 0.0066240595682091315, 'cat_smooth': 4.059611610484305}. Best is trial 0 with value: 0.8525826760216981.


... (70 outputs left out)

Best trial: 19. Best value: 0.847885:  98%|█████████▊| 39/40 [14:32<00:14, 14.17s/it]

[I 2026-10-30 21:19:14,556] Trial 38 pruned. 


Best trial: 19. Best value: 0.847885: 100%|██████████| 40/40 [14:40<00:00, 22.02s/it]

[I 2026-10-30 21:19:22,526] Trial 39 pruned. 


In [9]:
print('лучший RMSE:', round(study.best_value, 4))
print('best_iteration:', study.best_trial.user_attrs['best_iteration'])
study.best_params

лучший RMSE: 0.8479
best_iteration: 222


{'objective': 'regression',
 'num_leaves': 84,
 'min_child_samples': 236,
 'colsample_bytree': 0.6645097995051339,
 'subsample': 0.5756895436932168,
 'reg_lambda': 2.8570139411689,
 'cat_smooth': 5.178938561126461}

In [10]:
trials = study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'user_attrs'))
trials.sort_values('value').head(10)

,number,value,state,params_cat_smooth,params_colsample_bytree,params_min_child_samples,params_num_leaves,params_objective,params_reg_lambda,params_subsample,params_tweedie_variance_power,user_attrs_best_iteration
19,19,0.847885,COMPLETE,5.178939,0.664510,236,84,regression,2.857014,0.575690,NaN,222.0
22,22,0.848093,COMPLETE,2.155426,0.531710,216,66,regression,0.355039,0.584604,NaN,74.0
36,36,0.848166,COMPLETE,2.635678,0.635233,190,53,regression,2.503840,0.553793,NaN,256.0
11,11,0.849219,COMPLETE,4.593161,0.482340,378,101,regression,0.001509,0.901650,NaN,140.0
25,25,0.849348,COMPLETE,1.880862,0.666202,182,76,regression,23.678440,0.558538,NaN,102.0
24,24,0.850457,COMPLETE,1.200263,0.543814,93,85,regression,11.790332,0.572029,NaN,70.0
27,27,0.850539,PRUNED,2.754658,0.699206,389,60,regression,1.337053,0.650597,NaN,NaN
18,18,0.850611,COMPLETE,5.603752,0.549499,212,57,regression,0.097359,0.546068,NaN,77.0
30,30,0.851068,COMPLETE,5.791823,0.633235,192,97,regression,1.610542,0.582829,NaN,56.0
20,20,0.851228,COMPLETE,2.971402,0.572738,185,96,regression,0.015046,0.741278,NaN,106.0


In [11]:
optuna.visualization.plot_optimization_history(study).show()
optuna.visualization.plot_param_importances(study).show()

c:\Users\Admin\Desktop\Projects\ml-tabular-pfs\.venv\Lib\site-packages\optuna\importance\_ped_anova\evaluator.py:353: UserWarning: Some regimes for parameter `tweedie_variance_power` have less than 2 trials. The importance of the parameter may be inaccurate.
  optuna_warn(


In [12]:
optuna.visualization.plot_slice(
    study, params=['num_leaves', 'min_child_samples', 'colsample_bytree', 'subsample', 'reg_lambda']
).show()

In [13]:
best_params = {**FIXED, **study.best_params, 'learning_rate': 0.05, 'n_estimators': 5000}

model = lgb.LGBMRegressor(**best_params)
model.fit(
    train[FEATURES], train['target'],
    eval_X=(tr_s[FEATURES], valid[FEATURES]),
    eval_y=(tr_s['target'], valid['target']),
    eval_names=['train', 'valid'],
    categorical_feature=CAT_FEATURES,
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)],
)
print('best_iteration:', model.best_iteration_)

Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 0.78658	valid's rmse: 0.851909
[200]	train's rmse: 0.767083	valid's rmse: 0.85062
[300]	train's rmse: 0.756163	valid's rmse: 0.851027
[400]	train's rmse: 0.748532	valid's rmse: 0.851171
Early stopping, best iteration is:
[215]	train's rmse: 0.765106	valid's rmse: 0.85042
best_iteration: 215


In [14]:
res = calculate_metrics(
    model,
    {'train': (train[FEATURES], train['target']), 'valid': (valid[FEATURES], valid['target'])},
    model_name='LightGBM (Optuna)',
    baselines={'zeros': np.zeros(len(valid)), 'naive': valid['target_lag_1'].fillna(0)},
    backend='plotly',
)
res.round(4)

,rmse,mae,bias,r2,rmse_nonzero
train,0.7789,0.2908,-0.0003,0.5764,1.8770
valid,0.8504,0.3027,0.0172,0.4693,2.0549
valid: zeros,1.1982,0.2701,-0.2701,-0.0536,3.2177
valid: naive,1.0766,0.3202,0.0106,0.1494,2.6185


In [15]:
# сохраняем, чтобы model.ipynb и сабмит брали параметры отсюда
best_params['n_estimators'] = model.best_iteration_
(OPTUNA_DIR / f'{STUDY_NAME}_best_params.json').write_text(json.dumps(best_params, indent=2))
best_params

{'n_estimators': 215,
 'learning_rate': 0.05,
 'metric': 'rmse',
 'subsample_freq': 1,
 'random_state': 42,
 'n_jobs': -1,
 'verbose': -1,
 'objective': 'regression',
 'num_leaves': 84,
 'min_child_samples': 236,
 'colsample_bytree': 0.6645097995051339,
 'subsample': 0.5756895436932168,
 'reg_lambda': 2.8570139411689,
 'cat_smooth': 5.178938561126461}